In [ ]:

# CELL 1: Install dependencies

%pip install -q -U \
    "transformers>=4.51.0" \
    "datasets>=3.0.0" \
    "trl>=0.20.0" \
    "peft>=0.15.0" \
    "accelerate>=1.0.0" \
    bitsandbytes \
    gradio \
    huggingface_hub

print("Installation completed. Check for dependency errors above.")


In [ ]:

# CELL 1A: Verify dependencies

import sys
import importlib.metadata as metadata

print("Python:", sys.version)

packages = [
    "torch",
    "transformers",
    "datasets",
    "trl",
    "peft",
    "accelerate",
    "bitsandbytes",
    "gradio",
    "huggingface_hub",
    "pyarrow",
    "cudf-cu13",
]

for package in packages:
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

print("\n--- Import tests ---")

for module in ["torch", "transformers", "datasets", "trl", "peft", "gradio"]:
    try:
        __import__(module)
        print(f"{module}: OK")
    except Exception as exc:
        print(f"{module}: ERROR - {type(exc).__name__}: {exc}")

print("\nCUDA available:", __import__("torch").cuda.is_available())


In [ ]:

# OPTIONAL: Run only if you need cuDF compatibility.
%pip install "pyarrow>=19,<24"


In [ ]:

# CELL 2A: Verify GPU and 4-bit quantization support

import torch
import bitsandbytes as bnb
from transformers import BitsAndBytesConfig

print("CUDA available:", torch.cuda.is_available())
print("PyTorch CUDA build:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"\nGPU {i}: {props.name}")
        print(f"Total memory: {props.total_memory / (1024**3):.2f} GiB")
        print(f"Compute capability: {props.major}.{props.minor}")

    try:
        config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True
        )
        print("\n4-bit configuration created successfully.")
        print("bitsandbytes version:", bnb.__version__)
    except Exception as exc:
        print("\n4-bit configuration error:", repr(exc))
else:
    print("No GPU available. Do not load Gemma 3 4B yet.")


In [ ]:

# CELL 3: Load Gemma 3 4B Instruct on the Tesla T4

import torch
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

MODEL_ID = "google/gemma-3-4b-it"

assert torch.cuda.is_available(), (
    "CUDA GPU unavailable. Select a GPU runtime first."
)

# Use 4-bit quantization to reduce GPU memory requirements.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Gemma tokenizer and processor...")

processor = AutoProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer

print("Loading the quantized model. This may take several minutes...")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
model.config.use_cache = True

print("\nModel loaded successfully!")
print("Model class:", type(model).__name__)
print("Tokenizer class:", type(tokenizer).__name__)
print("Model device:", model.device)
print(
    "GPU memory allocated (GiB):",
    round(torch.cuda.memory_allocated(0) / (1024**3), 2)
)
print(
    "GPU memory reserved (GiB):",
    round(torch.cuda.memory_reserved(0) / (1024**3), 2)
)


In [ ]:

# CELL 3A: Authenticate with Hugging Face

from huggingface_hub import login, whoami

login()

user_info = whoami()
print("Authenticated Hugging Face account:", user_info["name"])


In [ ]:

# CELL 3B: Verify gated model access

from huggingface_hub import HfApi

MODEL_ID = "google/gemma-3-4b-it"

try:
    info = HfApi().model_info(MODEL_ID)
    print("Model access verified.")
    print("Model:", info.id)
    print("You can now retry the model-loading cell.")
except Exception as exc:
    print("Model access verification failed.")
    print(type(exc).__name__ + ":", str(exc))
    print(
        "\nCheck that you accepted the Gemma terms, "
        "have access, and authenticated with the correct account."
    )


In [ ]:

# CELL 3C: Load Gemma 3 4B after successful authentication

import torch
from huggingface_hub import get_token
from transformers import (
    AutoProcessor,
    AutoModelForImageTextToText,
    BitsAndBytesConfig
)

MODEL_ID = "google/gemma-3-4b-it"

assert torch.cuda.is_available(), "CUDA GPU is unavailable."

hf_token = get_token()
assert hf_token, "Hugging Face token not found. Run login() again."

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading processor...")

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=hf_token
)

tokenizer = processor.tokenizer

print("Loading quantized Gemma 3 4B model...")

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quant_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

model.eval()
model.config.use_cache = True

print("\nModel loaded successfully!")
print("Model class:", type(model).__name__)
print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory allocated (GiB):",
    round(torch.cuda.memory_allocated(0) / (1024**3), 2)
)


In [ ]:

# CELL 4: Test Gemma 3 4B text generation

import torch

messages = [
    {
        "role": "user",
        "content": (
            "I cannot connect to the office VPN. "
            "Give me three safe troubleshooting steps."
        )
    }
]

print("Preparing input...")

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

# Place model inputs on the input embedding device
input_device = model.get_input_embeddings().weight.device

inputs = {
    key: value.to(input_device) if torch.is_tensor(value) else value
    for key, value in inputs.items()
}

prompt_length = inputs["input_ids"].shape[1]

print("Generating response...")

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        use_cache=True
    )

# Decode only the newly generated tokens
new_tokens = outputs[0][prompt_length:]

answer = tokenizer.decode(
    new_tokens,
    skip_special_tokens=True
).strip()

print("\n=== HelpDeskBuddy Generation Test ===")
print(answer)

print("\n=== GPU Memory ===")
print(
    "Allocated (GiB):",
    round(torch.cuda.memory_allocated(0) / (1024**3), 2)
)
print(
    "Reserved (GiB):",
    round(torch.cuda.memory_reserved(0) / (1024**3), 2)
)


In [ ]:

# CELL 4A: Diagnose why Gemma returned an empty answer

import torch

print("=== Input diagnostics ===")
print("Input keys:", list(inputs.keys()))
print("Input IDs shape:", tuple(inputs["input_ids"].shape))
print("Prompt length:", prompt_length)
print("Model class:", type(model).__name__)
print("Encoder-decoder model:", model.config.is_encoder_decoder)

print("\n=== Output diagnostics ===")
print("Output type:", type(outputs).__name__)
print("Output shape:", tuple(outputs.shape))
print("Output token count:", outputs.shape[-1])

generated_ids = outputs[0][prompt_length:]

print("Generated token count:", generated_ids.numel())
print("Generated token IDs:", generated_ids.tolist())

print("\n=== Decode full output ===")
print(repr(tokenizer.decode(
    outputs[0],
    skip_special_tokens=False
)))

print("\n=== Decode generated portion ===")
print("With special tokens:", repr(tokenizer.decode(
    generated_ids,
    skip_special_tokens=False
)))
print("Without special tokens:", repr(tokenizer.decode(
    generated_ids,
    skip_special_tokens=True
)))

print("\n=== Token configuration ===")
print("Tokenizer EOS ID:", tokenizer.eos_token_id)
print("Tokenizer EOS token:", repr(tokenizer.eos_token))
print("Model generation EOS ID:", model.generation_config.eos_token_id)
print("Model generation PAD ID:", model.generation_config.pad_token_id)


In [ ]:

# CELL 4B: Diagnose Gemma 3 next-token logits

import torch

model.eval()

print("Checking model output logits...")

with torch.inference_mode():
    result = model(
        **inputs,
        use_cache=False
    )

logits = result.logits[0, -1, :].float()

finite_mask = torch.isfinite(logits)

print("\n=== Logits health ===")
print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(finite_mask.all().item()))
print("NaN count:", int(torch.isnan(logits).sum().item()))
print("Positive infinity count:", int(torch.isposinf(logits).sum().item()))
print("Negative infinity count:", int(torch.isneginf(logits).sum().item()))

if finite_mask.any():
    finite_logits = logits[finite_mask]

    print("Minimum finite logit:", finite_logits.min().item())
    print("Maximum finite logit:", finite_logits.max().item())

    # Find the highest-scoring finite token IDs
    safe_logits = logits.masked_fill(~finite_mask, float("-inf"))
    top_values, top_ids = torch.topk(safe_logits, k=10)

    print("\n=== Top 10 predicted tokens ===")
    for rank, (token_id, score) in enumerate(
        zip(top_ids.tolist(), top_values.tolist()), start=1
    ):
        token_text = tokenizer.decode([token_id])
        print(
            f"{rank}. ID={token_id}, "
            f"token={token_text!r}, logit={score:.4f}"
        )

    pad_id = tokenizer.pad_token_id
    if pad_id is not None:
        print("\nPAD token ID:", pad_id)
        print("PAD token logit:", logits[pad_id].item())
        print(
            "PAD is top prediction:",
            int(top_ids[0].item()) == pad_id
        )
else:
    print("ERROR: No finite logits were produced.")

del result, logits
torch.cuda.empty_cache()

print("\nLogits diagnostic completed.")


In [ ]:

# CELL 4C: Locate the first non-finite activation

import torch

model.eval()
activation_status = {}
hooks = []

def inspect_output(name):
    def hook(module, module_input, module_output):
        tensors = []

        def collect(obj):
            if torch.is_tensor(obj) and obj.is_floating_point():
                tensors.append(obj)
            elif isinstance(obj, (tuple, list)):
                for item in obj:
                    collect(item)
            elif isinstance(obj, dict):
                for item in obj.values():
                    collect(item)

        collect(module_output)

        if not tensors:
            activation_status[name] = "No floating-point output"
            return

        bad = []
        for tensor in tensors:
            if not torch.isfinite(tensor).all().item():
                bad.append({
                    "shape": tuple(tensor.shape),
                    "nan": int(torch.isnan(tensor).sum().item()),
                    "pos_inf": int(torch.isposinf(tensor).sum().item()),
                    "neg_inf": int(torch.isneginf(tensor).sum().item())
                })

        activation_status[name] = (
            {"status": "NON-FINITE", "details": bad}
            if bad else {"status": "finite"}
        )

    return hook

# Monitor the embedding layer and every language transformer layer.
embedding_layer = model.language_model.embed_tokens
hooks.append(
    embedding_layer.register_forward_hook(
        inspect_output("embedding")
    )
)

for i, layer in enumerate(model.language_model.layers):
    hooks.append(
        layer.register_forward_hook(
            inspect_output(f"decoder_layer_{i}")
        )
    )

# Also inspect the final normalization layer if available.
final_norm = getattr(model.language_model, "norm", None)
if final_norm is not None:
    hooks.append(
        final_norm.register_forward_hook(
            inspect_output("final_norm")
        )
    )

try:
    with torch.inference_mode():
        diagnostic_output = model(
            **inputs,
            use_cache=False
        )

    logits = diagnostic_output.logits

    print("=== Final logits ===")
    print("All finite:", bool(torch.isfinite(logits).all().item()))
    print("NaN count:", int(torch.isnan(logits).sum().item()))

    print("\n=== Layer activation results ===")
    first_bad = None

    for name, status in activation_status.items():
        print(name, ":", status)

        if (
            isinstance(status, dict)
            and status.get("status") == "NON-FINITE"
            and first_bad is None
        ):
            first_bad = name

    print("\n=== Diagnosis ===")
    if first_bad:
        print("First monitored layer with non-finite output:", first_bad)
    else:
        print(
            "No monitored layer reported non-finite values. "
            "Check the final model output and runtime compatibility."
        )

finally:
    for hook in hooks:
        hook.remove()

    if "diagnostic_output" in locals():
        del diagnostic_output

    if "logits" in locals():
        del logits

    torch.cuda.empty_cache()


In [ ]:

# CELL 4C-1: Inspect Gemma 3 model structure

import torch

print("=== Model information ===")
print("Model class:", type(model).__name__)
print("Transformers version:")

import transformers
print(transformers.__version__)

print("\n=== Top-level model components ===")
for name, module in model.named_children():
    print(f"{name}: {type(module).__name__}")

print("\n=== Embedding and transformer-layer paths ===")
matches = []

for name, module in model.named_modules():
    if (
        "embed_tokens" in name
        or name.endswith("layers")
        or name.endswith("norm")
        or "language_model" in name
    ):
        matches.append((name, type(module).__name__))

for name, class_name in matches[:100]:
    print(f"{name or '<root>'}: {class_name}")

print("\n=== Model parameter devices and dtypes ===")
for name, parameter in list(model.named_parameters())[:12]:
    print(
        name,
        "| shape:", tuple(parameter.shape),
        "| dtype:", parameter.dtype,
        "| device:", parameter.device
    )


In [ ]:

# CELL 4C-2 RETRY: Find the first non-finite activation

import torch

print("RUNNING CELL 4C-2 — ACTIVATION DIAGNOSTIC")

model.eval()
text_model = model.model.language_model
results = {}
hooks = []

def check_activation(name):
    def hook(module, module_input, module_output):
        tensors = []

        def collect(obj):
            if torch.is_tensor(obj) and obj.is_floating_point():
                tensors.append(obj)
            elif isinstance(obj, (tuple, list)):
                for item in obj:
                    collect(item)
            elif isinstance(obj, dict):
                for item in obj.values():
                    collect(item)

        collect(module_output)

        results[name] = []
        for t in tensors:
            results[name].append({
                "shape": tuple(t.shape),
                "dtype": str(t.dtype),
                "finite": bool(torch.isfinite(t).all().item()),
                "nan_count": int(torch.isnan(t).sum().item())
            })

    return hook

try:
    hooks.append(
        text_model.embed_tokens.register_forward_hook(
            check_activation("embedding")
        )
    )

    for i, layer in enumerate(text_model.layers):
        hooks.append(
            layer.register_forward_hook(
                check_activation(f"layer_{i}")
            )
        )

    if hasattr(text_model, "norm"):
        hooks.append(
            text_model.norm.register_forward_hook(
                check_activation("final_norm")
            )
        )

    hooks.append(
        model.lm_head.register_forward_hook(
            check_activation("lm_head")
        )
    )

    with torch.inference_mode():
        check_output = model(**inputs, use_cache=False)

    print("\nFinal logits finite:",
          bool(torch.isfinite(check_output.logits).all().item()))
    print("Final logits NaN count:",
          int(torch.isnan(check_output.logits).sum().item()))

    print("\n=== Activation results ===")
    first_bad = None

    for name, tensors in results.items():
        is_bad = any(not item["finite"] for item in tensors)
        print(f"{name}: {'NON-FINITE' if is_bad else 'finite'}")

        if is_bad:
            for item in tensors:
                if not item["finite"]:
                    print("  ", item)
            if first_bad is None:
                first_bad = name

    print("\nFirst monitored module with non-finite output:", first_bad)

finally:
    for h in hooks:
        h.remove()

    if "check_output" in locals():
        del check_output

    torch.cuda.empty_cache()


In [ ]:

# CELL 4C-3: Inspect internal components of decoder layer 5

import torch

model.eval()

layer5 = model.model.language_model.layers[5]
module_results = {}
hooks = []

def inspect_module_output(name):
    def hook(module, module_input, module_output):
        tensors = []

        def collect(obj):
            if torch.is_tensor(obj) and obj.is_floating_point():
                tensors.append(obj)
            elif isinstance(obj, (tuple, list)):
                for item in obj:
                    collect(item)
            elif isinstance(obj, dict):
                for item in obj.values():
                    collect(item)

        collect(module_output)

        reports = []
        for t in tensors:
            finite = torch.isfinite(t)
            reports.append({
                "shape": tuple(t.shape),
                "dtype": str(t.dtype),
                "nan": int(torch.isnan(t).sum().item()),
                "positive_inf": int(torch.isposinf(t).sum().item()),
                "negative_inf": int(torch.isneginf(t).sum().item()),
                "finite": bool(finite.all().item()),
                "max_abs_finite": (
                    float(t[finite].abs().max().item())
                    if finite.any().item() else None
                )
            })

        module_results[name] = reports

    return hook

# Register hooks on layer 5 and all its nested components.
for name, module in layer5.named_modules():
    full_name = "layer5" + ("." + name if name else "")
    hooks.append(
        module.register_forward_hook(
            inspect_module_output(full_name)
        )
    )

try:
    with torch.inference_mode():
        diagnostic_output = model(**inputs, use_cache=False)

    print("=== Non-finite components inside layer 5 ===")

    found = False
    for name, reports in module_results.items():
        bad_reports = [r for r in reports if not r["finite"]]
        if bad_reports:
            found = True
            print(f"\n{name}")
            for report in bad_reports:
                print(report)

    if not found:
        print("No internal component output was marked non-finite.")

    print("\n=== Layer 5 component summary ===")
    for name, reports in module_results.items():
        if reports:
            print(
                name,
                "| finite:",
                all(r["finite"] for r in reports),
                "| tensors:", len(reports)
            )

finally:
    for hook in hooks:
        hook.remove()

    if "diagnostic_output" in locals():
        del diagnostic_output

    torch.cuda.empty_cache()


In [ ]:

# CELL 4C-4: Trace the overflow at decoder layer 5

import torch
import inspect

model.eval()
layer5 = model.model.language_model.layers[5]

layer_input_info = {}
layer_output_info = {}

def tensor_stats(t):
    t = t.detach().float()
    finite = torch.isfinite(t)
    return {
        "shape": tuple(t.shape),
        "finite": bool(finite.all().item()),
        "nan_count": int(torch.isnan(t).sum().item()),
        "positive_inf_count": int(torch.isposinf(t).sum().item()),
        "negative_inf_count": int(torch.isneginf(t).sum().item()),
        "max_abs_finite": (
            float(t[finite].abs().max().item())
            if finite.any().item() else None
        )
    }

def capture_input(module, args):
    x = args[0]
    if torch.is_tensor(x):
        layer_input_info.update(tensor_stats(x))

def capture_output(module, args, output):
    x = output[0] if isinstance(output, (tuple, list)) else output
    if torch.is_tensor(x):
        layer_output_info.update(tensor_stats(x))
        bad = ~torch.isfinite(x)
        if bad.any():
            print("First non-finite output indices:",
                  bad.nonzero()[:10].detach().cpu().tolist())
            print("Values at those indices:",
                  x[bad][:10].detach().cpu().tolist())

pre_hook = layer5.register_forward_pre_hook(capture_input)
post_hook = layer5.register_forward_hook(capture_output)

try:
    with torch.inference_mode():
        _ = model(**inputs, use_cache=False)

    print("=== Layer 5 input ===")
    print(layer_input_info)

    print("\n=== Layer 5 output ===")
    print(layer_output_info)

finally:
    pre_hook.remove()
    post_hook.remove()
    torch.cuda.empty_cache()

print("\n=== Layer 5 forward implementation ===")
try:
    print(inspect.getsource(type(layer5).forward))
except (OSError, TypeError) as e:
    print("Could not retrieve forward source:", e)
    print("Forward signature:", inspect.signature(layer5.forward))


In [ ]:

# CELL 4C-5: Identify which residual addition overflows

import torch

model.eval()
layer5 = model.model.language_model.layers[5]

captured = {}
hooks = []

def save_input(name):
    def hook(module, args):
        if args and torch.is_tensor(args[0]):
            captured[name] = args[0].detach().clone()
    return hook

def save_output(name):
    def hook(module, args, output):
        x = output[0] if isinstance(output, (tuple, list)) else output
        if torch.is_tensor(x):
            captured[name] = x.detach().clone()
    return hook

hooks.append(layer5.register_forward_pre_hook(save_input("layer_input")))

for name in [
    "post_attention_layernorm",
    "pre_feedforward_layernorm",
    "post_feedforward_layernorm",
]:
    module = getattr(layer5, name)
    hooks.append(module.register_forward_hook(save_output(name)))

try:
    with torch.inference_mode():
        _ = model(**inputs, use_cache=False)

    # Residual 1: layer input + attention output after normalization
    residual1_fp16 = (
        captured["layer_input"] + captured["post_attention_layernorm"]
    )
    residual1_fp32 = (
        captured["layer_input"].float()
        + captured["post_attention_layernorm"].float()
    )

    # Residual 2: residual 1 + feedforward output after normalization
    residual2_fp16 = (
        residual1_fp16 + captured["post_feedforward_layernorm"]
    )
    residual2_fp32 = (
        residual1_fp32
        + captured["post_feedforward_layernorm"].float()
    )

    def report(name, x16, x32):
        print(f"\n=== {name} ===")
        print("FP16 finite:", bool(torch.isfinite(x16).all().item()))
        print("FP16 +Inf:", int(torch.isposinf(x16).sum().item()))
        print("FP16 -Inf:", int(torch.isneginf(x16).sum().item()))
        print("FP32 finite:", bool(torch.isfinite(x32).all().item()))
        print("FP32 max abs:", float(x32.abs().max().item()))
        bad = ~torch.isfinite(x16)
        if bad.any():
            print("First FP16 non-finite indices:",
                  bad.nonzero()[:10].detach().cpu().tolist())
            print("Corresponding FP32 values:",
                  x32[bad][:10].detach().cpu().tolist())

    report("First residual addition", residual1_fp16, residual1_fp32)
    report("Second residual addition", residual2_fp16, residual2_fp32)

finally:
    for hook in hooks:
        hook.remove()
    torch.cuda.empty_cache()


In [ ]:

# CELL 4C-6: Reload Gemma 3 4B with FP32 computation

import gc
import torch
from transformers import AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-4b-it"

# Release the existing model before reloading.
try:
    del model
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Retain 4-bit weight quantization, but use FP32 computation.
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float32,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    token=hf_token,
    quantization_config=quant_config,
    dtype=torch.float32,
    device_map="auto",
    low_cpu_mem_usage=True,
)

model.eval()
model.config.use_cache = True

print("\nModel loaded.")
print("Model class:", type(model).__name__)
print("Embedding dtype:", model.get_input_embeddings().weight.dtype)
print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")


In [ ]:

# CELL 4C-7: Verify finite logits before generation

import torch

messages = [
    {
        "role": "user",
        "content": "I cannot connect to the office VPN. Give me three safe troubleshooting steps."
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

input_device = model.get_input_embeddings().weight.device

inputs = {
    key: value.to(input_device) if torch.is_tensor(value) else value
    for key, value in inputs.items()
}

with torch.inference_mode():
    result = model(**inputs, use_cache=False)

logits = result.logits.float()

print("Logits shape:", tuple(logits.shape))
print("All logits finite:", bool(torch.isfinite(logits).all().item()))
print("NaN count:", int(torch.isnan(logits).sum().item()))
print("Positive infinity count:", int(torch.isposinf(logits).sum().item()))
print("Negative infinity count:", int(torch.isneginf(logits).sum().item()))

del result, logits
torch.cuda.empty_cache()


In [ ]:

# CELL 4C-8: Test Gemma 3 4B text generation

import torch

model.eval()

messages = [
    {
        "role": "user",
        "content": (
            "I cannot connect to the office VPN. "
            "Give me three safe troubleshooting steps."
        )
    }
]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
)

input_device = model.get_input_embeddings().weight.device

inputs = {
    key: value.to(input_device) if torch.is_tensor(value) else value
    for key, value in inputs.items()
}

prompt_length = inputs["input_ids"].shape[1]

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

generated_tokens = outputs[0, prompt_length:]

print("Prompt tokens:", prompt_length)
print("Generated tokens:", generated_tokens.numel())
print("Generated token IDs:", generated_tokens[:20].tolist())

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True,
).strip()

print("\n=== HelpDeskBuddy Base Model Response ===")
print(answer if answer else "[Empty response]")
print("\nGPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GiB")


In [ ]:

# CELL 5: Validate the HelpDeskBuddy JSONL dataset

import os
import json

DATASET_PATH = "/content/helpdesk_dataset.jsonl"

if not os.path.isfile(DATASET_PATH):
    raise FileNotFoundError(
        f"Dataset not found: {DATASET_PATH}\n"
        "Upload helpdesk_dataset.jsonl to the Colab session first."
    )

valid_records = 0
invalid_records = []
required_fields = {"instruction", "response"}

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as e:
            invalid_records.append(
                f"Line {line_number}: Invalid JSON — {e}"
            )
            continue

        if not isinstance(record, dict):
            invalid_records.append(
                f"Line {line_number}: Record must be a JSON object."
            )
            continue

        missing = required_fields - record.keys()

        if missing:
            invalid_records.append(
                f"Line {line_number}: Missing fields {sorted(missing)}"
            )
        elif not isinstance(record["instruction"], str) or not record["instruction"].strip():
            invalid_records.append(
                f"Line {line_number}: Empty or invalid instruction."
            )
        elif not isinstance(record["response"], str) or not record["response"].strip():
            invalid_records.append(
                f"Line {line_number}: Empty or invalid response."
            )
        else:
            valid_records += 1

print("Dataset path:", DATASET_PATH)
print("File size:", os.path.getsize(DATASET_PATH), "bytes")
print("Valid records:", valid_records)
print("Invalid records:", len(invalid_records))

for issue in invalid_records[:20]:
    print("ERROR:", issue)

if valid_records == 0:
    raise ValueError("No valid instruction/response records found.")

if invalid_records:
    print("\nFix the reported issues before training.")
else:
    print("\nDataset structure validation passed.")

    with open(DATASET_PATH, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                sample = json.loads(line)
                print("\n=== Sample instruction ===")
                print(sample["instruction"])
                print("\n=== Sample response ===")
                print(sample["response"])
                break


In [ ]:

# CELL 5A: Search common Colab locations for the dataset

import os

TARGET = "helpdesk_dataset.jsonl"
SEARCH_ROOTS = ["/content", "/tmp"]

matches = []

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue

    for current_dir, dirs, files in os.walk(root):
        # Avoid scanning cache and system-heavy directories.
        dirs[:] = [
            d for d in dirs
            if d not in {" .cache", ".cache", "__pycache__", "node_modules"}
        ]

        if TARGET in files:
            matches.append(os.path.join(current_dir, TARGET))

print("Dataset matches:")
if matches:
    for path in matches:
        print(path)
else:
    print("No matching file found in /content or /tmp.")

print("\nTop-level /content items:")
for item in os.listdir("/content"):
    print(item)


In [ ]:

# CELL 5B: Mount Google Drive and locate the dataset

from google.colab import drive
import os

drive.mount("/content/drive")

TARGET = "helpdesk_dataset.jsonl"
matches = []

for current_dir, dirs, files in os.walk("/content/drive/MyDrive"):
    if TARGET in files:
        matches.append(os.path.join(current_dir, TARGET))

print("\nDataset matches in Google Drive:")
if matches:
    for path in matches:
        print(path)
else:
    print("Dataset not found in Google Drive.")


In [ ]:

# CELL 5C: Upload the existing JSONL dataset from your computer

from google.colab import files
import os
import shutil

uploaded = files.upload()

if "helpdesk_dataset.jsonl" not in uploaded:
    raise ValueError(
        "Please select the file named helpdesk_dataset.jsonl."
    )

source = "/content/helpdesk_dataset.jsonl"
print("Dataset available:", os.path.isfile(source))
print("File size:", os.path.getsize(source), "bytes")


In [ ]:

# CELL 5D: Create a synthetic HelpDeskBuddy IT-support dataset

import json
import os

DATASET_PATH = "/content/helpdesk_dataset.jsonl"

dataset = [
    {
        "instruction": "I cannot connect to the office VPN.",
        "response": "1. Check that your internet connection works. 2. Verify the VPN server address and reconnect using your organization's approved VPN client. 3. Restart the client and check for error messages. If the issue persists, contact IT with the error code. Never share your password or MFA code."
    },
    {
        "instruction": "My Windows computer cannot connect to Wi-Fi.",
        "response": "1. Confirm Wi-Fi is enabled and airplane mode is off. 2. Check whether other devices can connect to the same network. 3. Disconnect and reconnect to the approved network. If the problem continues, report the error and device name to IT."
    },
    {
        "instruction": "I can connect to Wi-Fi but websites do not open.",
        "response": "1. Check whether multiple websites are affected. 2. Confirm other devices can access the internet. 3. Restart the browser and check the network connection. If the issue persists, ask IT to investigate DNS, proxy, or network restrictions."
    },
    {
        "instruction": "Outlook is not receiving new emails.",
        "response": "1. Check your internet connection and Outlook connection status. 2. Confirm Outlook is not in Work Offline mode. 3. Check mailbox storage and the Junk Email folder. If the issue persists, contact IT with the approximate time the problem started."
    },
    {
        "instruction": "Microsoft Teams microphone is not working.",
        "response": "1. Check that the microphone is connected and not muted. 2. In Teams settings, select the correct microphone and run a test call if available. 3. Check Windows microphone permissions. If the device is managed and settings are restricted, contact IT."
    },
    {
        "instruction": "My printer is showing as offline.",
        "response": "1. Check that the printer is powered on and connected to the network or computer. 2. Confirm you selected the correct printer. 3. Review the print queue for stuck jobs. If it remains offline, provide IT with the printer name and any displayed error."
    },
    {
        "instruction": "My Windows computer is running very slowly.",
        "response": "1. Save your work and restart the computer if practical. 2. Check Task Manager for unusually high CPU, memory, or disk usage. 3. Ensure sufficient disk space is available. Do not terminate unfamiliar security or system processes; contact IT if performance remains poor."
    },
    {
        "instruction": "I cannot sign in to my Windows domain account.",
        "response": "1. Check the username format and keyboard layout. 2. Confirm network connectivity and, if applicable, connect to the organization's VPN. 3. Note the exact error message. Do not repeatedly guess passwords; contact the authorized IT team if the account may be locked."
    },
    {
        "instruction": "A user account may be locked in Active Directory.",
        "response": "1. Verify the user's identity using the organization's approved procedure. 2. Check the account status using authorized administrative tools. 3. Follow the organization's account-unlock process and investigate repeated lockouts. Never request or record the user's password."
    },
    {
        "instruction": "How can I troubleshoot a possible DNS issue?",
        "response": "1. Confirm the device has a valid network connection. 2. Compare access using a hostname with access to a known approved internal resource, where permitted. 3. An authorized technician can inspect DNS server settings and run nslookup. Escalate with the hostname, error, and affected network."
    },
    {
        "instruction": "My computer has run out of disk space.",
        "response": "1. Check Windows Storage settings to identify large files and temporary files. 2. Use approved cleanup tools and remove only files you recognize and no longer need. 3. Do not delete system folders or business records. Contact IT if storage remains critically low."
    },
    {
        "instruction": "Windows Update keeps failing.",
        "response": "1. Record the Windows Update error code. 2. Check network connectivity and available disk space. 3. Restart the computer when safe and retry according to company policy. Do not bypass enterprise update controls; escalate persistent failures to IT."
    },
    {
        "instruction": "My laptop battery is not charging.",
        "response": "1. Check the power outlet, adapter, and cable for visible damage. 2. Confirm the charger is compatible with the laptop. 3. Restart the device and review the battery status. Stop using damaged equipment and contact IT if charging still fails."
    },
    {
        "instruction": "My external monitor displays no signal.",
        "response": "1. Confirm the monitor is powered on and the correct input is selected. 2. Reseat the video cable and check the docking station connections. 3. Use Windows display settings to detect displays. If the problem persists, test approved equipment or contact IT."
    },
    {
        "instruction": "My keyboard or mouse has stopped responding.",
        "response": "1. Check the cable, USB port, battery, or wireless receiver. 2. If permitted, test another USB port or restart the computer. 3. Check whether the device works on another approved computer. Report the device model and symptoms if it still fails."
    },
    {
        "instruction": "I cannot access a shared network folder.",
        "response": "1. Confirm you are connected to the corporate network or approved VPN. 2. Verify the shared folder path and check whether colleagues are affected. 3. Record the exact access-denied or network error. Request access through the approved process rather than bypassing permissions."
    },
    {
        "instruction": "My computer has no sound.",
        "response": "1. Check the volume and mute settings. 2. Select the correct output device in Windows sound settings. 3. Reconnect the headset or speakers and test again. If the issue continues, contact IT with the device model and any audio error."
    },
    {
        "instruction": "A blue screen appeared on my Windows computer.",
        "response": "1. Record the stop code or photograph the error if company policy permits. 2. Allow the computer to restart and note whether the problem repeats. 3. Report recent hardware or software changes to IT. Do not repeatedly force shutdowns unless required to recover a completely unresponsive device."
    },
    {
        "instruction": "My computer cannot find a network printer.",
        "response": "1. Confirm you are connected to the correct corporate network. 2. Check the printer name and availability with your IT team. 3. Use the organization's approved printer installation procedure. Do not install unapproved drivers or change server settings without authorization."
    },
    {
        "instruction": "My browser displays a certificate warning on an internal website.",
        "response": "1. Check that the website address is correct. 2. Do not bypass the certificate warning or enter credentials on the affected page. 3. Capture the warning details and report the URL to IT or security through an approved channel."
    },
    {
        "instruction": "I received a suspicious email asking me to sign in urgently.",
        "response": "1. Do not click links or open unexpected attachments. 2. Do not enter credentials or share verification codes. 3. Report the email using your organization's phishing-reporting procedure. If you already entered credentials, contact IT or security immediately."
    },
    {
        "instruction": "My laptop is overheating and its fan is loud.",
        "response": "1. Place the laptop on a hard, unobstructed surface. 2. Save work and close unnecessary applications. 3. If the device becomes excessively hot, shuts down unexpectedly, or smells unusual, stop using it and contact IT. Do not open the device unless authorized."
    },
    {
        "instruction": "An application keeps crashing.",
        "response": "1. Record the application name and any error message. 2. Restart the application and check for approved updates. 3. Note whether the crash happens during a particular action. Contact IT with the details; do not disable security software to troubleshoot."
    },
    {
        "instruction": "My computer clock is incorrect.",
        "response": "1. Check the displayed date, time, and time zone. 2. Use the standard Windows time synchronization option if permitted. 3. On a domain-managed device, report persistent time or synchronization errors to IT because time differences can affect authentication."
    },
    {
        "instruction": "A user says their computer is disconnected from the corporate network.",
        "response": "1. Check the Ethernet cable or Wi-Fi connection. 2. Confirm whether other devices or users are affected. 3. Record the device name, connection type, and error message. An authorized technician can then inspect IP configuration, DHCP, switch connectivity, or network access controls."
    }
]

with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for record in dataset:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

# Verify the generated JSONL file.
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    verified_records = [json.loads(line) for line in f if line.strip()]

assert len(verified_records) == len(dataset)
assert all(
    isinstance(r.get("instruction"), str)
    and r["instruction"].strip()
    and isinstance(r.get("response"), str)
    and r["response"].strip()
    for r in verified_records
)

print("Dataset created successfully.")
print("Path:", DATASET_PATH)
print("Records:", len(verified_records))
print("File size:", os.path.getsize(DATASET_PATH), "bytes")
print("Fields:", list(verified_records[0].keys()))
print("\nSample instruction:", verified_records[0]["instruction"])
print("Sample response:", verified_records[0]["response"])


In [ ]:

# CELL 5E: Final dataset verification

import json
import os

DATASET_PATH = "/content/helpdesk_dataset.jsonl"

assert os.path.isfile(DATASET_PATH), "Dataset file is missing."

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

assert len(records) > 0
assert all(
    isinstance(r.get("instruction"), str)
    and isinstance(r.get("response"), str)
    and r["instruction"].strip()
    and r["response"].strip()
    for r in records
)

print("Dataset validation: PASSED")
print("Total records:", len(records))
print("Dataset size:", os.path.getsize(DATASET_PATH), "bytes")


In [ ]:

# CELL 6: Convert the dataset into Gemma chat-formatted examples

import json
from datasets import Dataset

DATASET_PATH = "/content/helpdesk_dataset.jsonl"

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

SYSTEM_PROMPT = (
    "You are HelpDeskBuddy, an IT support troubleshooting assistant. "
    "Give concise, safe, step-by-step guidance. Never ask users to share "
    "passwords, MFA codes, or secrets. Do not claim to have performed "
    "actions. Refer administrative or organization-specific changes to "
    "authorized IT staff."
)

formatted_records = []

for record in records:
    messages = [
        {
            "role": "user",
            "content": (
                SYSTEM_PROMPT
                + "\n\nUser issue: "
                + record["instruction"]
            ),
        },
        {
            "role": "assistant",
            "content": record["response"],
        },
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    formatted_records.append({"text": text})

train_dataset = Dataset.from_list(formatted_records)

print("Formatted training examples:", len(train_dataset))
print("\nSample formatted example:\n")
print(train_dataset[0]["text"][:1500])


In [ ]:

# CELL 5A: Search common Colab locations for the dataset

import os

TARGET = "helpdesk_dataset.jsonl"
SEARCH_ROOTS = ["/content", "/tmp"]

matches = []

for root in SEARCH_ROOTS:
    if not os.path.exists(root):
        continue

    for current_dir, dirs, files in os.walk(root):
        # Avoid scanning cache and system-heavy directories.
        dirs[:] = [
            d for d in dirs
            if d not in {" .cache", ".cache", "__pycache__", "node_modules"}
        ]

        if TARGET in files:
            matches.append(os.path.join(current_dir, TARGET))

print("Dataset matches:")
if matches:
    for path in matches:
        print(path)
else:
    print("No matching file found in /content or /tmp.")

print("\nTop-level /content items:")
for item in os.listdir("/content"):
    print(item)


In [ ]:

# CELL 5B: Mount Google Drive and locate the dataset

from google.colab import drive
import os

drive.mount("/content/drive")

TARGET = "helpdesk_dataset.jsonl"
matches = []

for current_dir, dirs, files in os.walk("/content/drive/MyDrive"):
    if TARGET in files:
        matches.append(os.path.join(current_dir, TARGET))

print("\nDataset matches in Google Drive:")
if matches:
    for path in matches:
        print(path)
else:
    print("Dataset not found in Google Drive.")


In [ ]:
from google.colab import files
files.download("/content/helpdesk_dataset.jsonl")

In [ ]:

# CELL 7: Inspect the installed training API and GPU memory

import inspect
import torch
import transformers
import trl
import peft
from trl import SFTTrainer, SFTConfig

print("=== Installed package versions ===")
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("PEFT:", peft.__version__)

print("\n=== Model and dataset ===")
print("Model class:", type(model).__name__)
print("Dataset rows:", len(train_dataset))
print("Dataset columns:", train_dataset.column_names)
print("Model device:", model.get_input_embeddings().weight.device)
print("Model dtype:", model.get_input_embeddings().weight.dtype)

print("\n=== SFTTrainer parameters ===")
print(inspect.signature(SFTTrainer))

print("\n=== SFTConfig parameters ===")
print(inspect.signature(SFTConfig))

print("\n=== GPU memory ===")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GiB"
    )
    print(
        "Reserved:",
        round(torch.cuda.memory_reserved(0) / 1024**3, 2),
        "GiB"
    )
else:
    print("CUDA GPU is not available.")


In [ ]:

# CELL 8: Verify LoRA target modules in Gemma 3 4B

import torch
from collections import Counter

TARGET_MODULES = [
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
]

module_counts = Counter()
module_examples = {}

for name, module in model.named_modules():
    suffix = name.rsplit(".", 1)[-1]

    if suffix in TARGET_MODULES:
        module_counts[suffix] += 1
        module_examples.setdefault(suffix, name)

print("=== LoRA target-module verification ===")

for target in TARGET_MODULES:
    print(
        f"{target}: {module_counts[target]} module(s)",
        f"| Example: {module_examples.get(target, 'NOT FOUND')}"
    )

missing = [
    target for target in TARGET_MODULES
    if module_counts[target] == 0
]

print("\n=== Quantization status ===")
print("Model class:", type(model).__name__)
print("Loaded in 4-bit:", getattr(model, "is_loaded_in_4bit", "not exposed"))
print("Gradient checkpointing currently enabled:",
      getattr(model, "is_gradient_checkpointing", False))

if missing:
    print("\nWARNING: These target modules were not found:", missing)
else:
    print("\nLoRA target-module verification: PASSED")

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GiB"
    )


In [ ]:

# CELL 9: Prepare Gemma 3 4B for language-only LoRA fine-tuning

import torch
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

# 1. Find supported projection modules in the language model only
TARGET_SUFFIXES = {
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
    "gate_proj",
    "up_proj",
    "down_proj",
}

LANGUAGE_PREFIX = "model.language_model.layers."

language_target_modules = [
    name
    for name, module in model.named_modules()
    if name.startswith(LANGUAGE_PREFIX)
    and name.rsplit(".", 1)[-1] in TARGET_SUFFIXES
]

assert language_target_modules, (
    "No language-model LoRA targets found. Stop and inspect model.named_modules()."
)

print("Language-model target modules:", len(language_target_modules))
print("Example targets:")
for name in language_target_modules[:5]:
    print(" ", name)

# 2. Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=language_target_modules,
    task_type="CAUSAL_LM",
)

# 3. Prepare the existing 4-bit model for training
if not getattr(model, "is_loaded_in_4bit", False):
    raise RuntimeError(
        "The model is not reported as 4-bit quantized. Stop before training."
    )

model.config.use_cache = False

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

# 4. Attach LoRA adapters
model = get_peft_model(model, lora_config)

# 5. Verify the result
model.print_trainable_parameters()

trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)
total_params = sum(p.numel() for p in model.parameters())

print("\n=== LoRA preparation results ===")
print("Model class:", type(model).__name__)
print("Trainable parameters:", f"{trainable_params:,}")
print("Total parameters:", f"{total_params:,}")
print(
    "Trainable percentage:",
    f"{100 * trainable_params / total_params:.4f}%"
)
print("Gradient checkpointing:", model.is_gradient_checkpointing)
print("Cache enabled:", model.config.use_cache)

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GiB"
    )

print("\nLoRA preparation cell completed.")


In [ ]:

# CELL 10: Configure SFTTrainer for memory-conscious LoRA fine-tuning

import torch
from trl import SFTConfig, SFTTrainer

OUTPUT_DIR = "/content/deskbuddy_gemma3_lora"

# Confirm the dataset and existing LoRA setup
assert len(train_dataset) > 0, "Training dataset is empty."
assert "text" in train_dataset.column_names, "Missing text column."
assert hasattr(model, "peft_config"), "LoRA adapters are not attached."

# Keep training settings conservative for the Tesla T4
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,

    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    warmup_steps=1,
    weight_decay=0.0,
    max_grad_norm=0.3,

    optim="adamw_torch",
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    use_cache=False,

    dataset_text_field="text",
    max_length=256,
    packing=False,
    completion_only_loss=False,
    assistant_only_loss=False,

    logging_strategy="steps",
    logging_steps=1,
    save_strategy="epoch",
    save_total_limit=1,
    save_only_model=True,

    report_to="none",
    remove_unused_columns=False,
    dataloader_num_workers=0,
    seed=42,
)

print("Initializing SFTTrainer...")

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

print("\n=== Trainer initialization successful ===")
print("Training examples:", len(train_dataset))
print("Epochs:", training_args.num_train_epochs)
print("Batch size per device:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Maximum sequence length:", training_args.max_length)
print("Output directory:", OUTPUT_DIR)
print("Training steps planned:", trainer.state.max_steps)

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GiB"
    )


In [ ]:

# CELL 10A: Check retained examples, labels, and planned batches

print("=== Trainer dataset diagnostics ===")

print("Original formatted examples:", len(train_dataset))
print("Trainer dataset examples:", len(trainer.train_dataset))
print("Trainer dataset columns:", trainer.train_dataset.column_names)

# Check how many batches the trainer can create
try:
    train_loader = trainer.get_train_dataloader()
    print("Training batches per epoch:", len(train_loader))
except Exception as exc:
    train_loader = None
    print("Dataloader creation failed:")
    print(type(exc).__name__ + ":", str(exc)[:1500])

# Inspect available token and label fields
if len(trainer.train_dataset) > 0:
    sample = trainer.train_dataset[0]

    print("\n=== First processed example ===")
    print("Available fields:", list(sample.keys()))

    for key in ("input_ids", "labels", "attention_mask"):
        if key in sample:
            value = sample[key]
            print(key, "length:", len(value))

    if "labels" in sample:
        labels = sample["labels"]
        valid_labels = sum(
            1 for token_id in labels if token_id != -100
        )
        print("Trainable label tokens in first example:", valid_labels)
        print("Masked label tokens (-100):", len(labels) - valid_labels)

    # Check several examples rather than just one
    if "labels" in trainer.train_dataset.column_names:
        print("\n=== Label check across examples ===")
        check_count = min(5, len(trainer.train_dataset))

        for i in range(check_count):
            labels = trainer.train_dataset[i]["labels"]
            valid = sum(1 for token_id in labels if token_id != -100)
            print(f"Example {i}: {valid} trainable label tokens out of {len(labels)}")

print("\n=== Trainer state ===")
print("State max_steps:", trainer.state.max_steps)
print("Configured max_steps:", trainer.args.max_steps)
print("Configured epochs:", trainer.args.num_train_epochs)


In [ ]:

# CELL 11: Train HelpDeskBuddy for one epoch

import os
import math
import torch

OUTPUT_DIR = "/content/deskbuddy_gemma3_lora"
ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")

# Preflight checks
assert len(trainer.train_dataset) == 25, (
    f"Expected 25 examples, found {len(trainer.train_dataset)}"
)

train_loader = trainer.get_train_dataloader()
assert len(train_loader) > 0, "No training batches are available."

print("=== Training preflight ===")
print("Training examples:", len(trainer.train_dataset))
print("Training batches:", len(train_loader))
print("Epochs:", trainer.args.num_train_epochs)
print("Adapter output:", ADAPTER_DIR)

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(
        "GPU memory allocated before training:",
        round(torch.cuda.memory_allocated(0) / 1024**3, 2),
        "GiB"
    )

# Start training
print("\n=== Starting fine-tuning ===")
train_result = trainer.train()

# Validate reported loss
train_loss = train_result.metrics.get("train_loss")

print("\n=== Training results ===")
print("Training loss:", train_loss)
print("Training metrics:", train_result.metrics)

if train_loss is None or not math.isfinite(float(train_loss)):
    raise RuntimeError(
        "Training did not produce a finite loss. Do not treat this run as successful."
    )

# Save the trained LoRA adapter
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("\n=== Saved adapter verification ===")
print("Adapter directory exists:", os.path.isdir(ADAPTER_DIR))

for filename in sorted(os.listdir(ADAPTER_DIR)):
    path = os.path.join(ADAPTER_DIR, filename)
    if os.path.isfile(path):
        print(f"{filename}: {os.path.getsize(path):,} bytes")

assert os.path.isfile(os.path.join(ADAPTER_DIR, "adapter_config.json")), (
    "Adapter configuration was not saved."
)

print("\nFine-tuning completed and adapter configuration verified.")


In [ ]:

# CELL 12: Test the fine-tuned HelpDeskBuddy model

import os
import torch

ADAPTER_DIR = "/content/deskbuddy_gemma3_lora/final_adapter"

assert os.path.isfile(
    os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
), "Saved LoRA adapter weights were not found."

model.eval()
model.config.use_cache = True

SYSTEM_PROMPT = (
    "You are HelpDeskBuddy, an IT support troubleshooting assistant. "
    "Give concise, safe, step-by-step guidance. Never ask users to share "
    "passwords, MFA codes, or secrets. Do not claim to have performed "
    "actions. Refer administrative or organization-specific changes to "
    "authorized IT staff."
)

TEST_QUESTIONS = [
    "I cannot connect to the office VPN. Give me three troubleshooting steps.",
    "My Outlook is not receiving new emails. What should I check?",
    "My computer cannot access an internal website. How can I troubleshoot DNS?"
]

def test_helpdeskbuddy(question):
    messages = [
        {
            "role": "user",
            "content": SYSTEM_PROMPT + "\n\nUser issue: " + question,
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    input_device = model.get_input_embeddings().weight.device
    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
        if torch.is_tensor(value)
    }

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = inputs["input_ids"].shape[-1]
    new_tokens = outputs[0, prompt_length:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

print("=== HelpDeskBuddy inference tests ===")

for index, question in enumerate(TEST_QUESTIONS, start=1):
    print(f"\nTEST {index}: {question}")
    try:
        answer = test_helpdeskbuddy(question)
        print("HelpDeskBuddy:", answer if answer else "[Empty response]")
    except Exception as exc:
        print("TEST FAILED:", type(exc).__name__, str(exc)[:1200])
        raise

print("\nInference tests completed.")


In [ ]:

# CELL 12A: Retest HelpDeskBuddy with longer answers

import os
import torch

ADAPTER_DIR = "/content/deskbuddy_gemma3_lora/final_adapter"

assert os.path.isfile(
    os.path.join(ADAPTER_DIR, "adapter_model.safetensors")
), "Saved adapter weights were not found."

model.eval()
model.config.use_cache = True

SYSTEM_PROMPT = (
    "You are HelpDeskBuddy, an IT support troubleshooting assistant. "
    "Give accurate, concise, numbered steps. Do not invent menu options "
    "or claim actions were performed. Never request passwords or MFA codes. "
    "Refer administrative changes to authorized IT staff."
)

TEST_QUESTIONS = [
    "I cannot connect to the office VPN. Give me three troubleshooting steps.",
    "My Outlook is not receiving new emails. What should I check?",
    "My computer cannot access an internal website. Explain safe DNS troubleshooting."
]

def test_helpdeskbuddy(question):
    messages = [{
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\nUser issue: " + question
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    input_device = model.get_input_embeddings().weight.device

    # Pass only the standard text-generation inputs.
    generation_inputs = {
        key: inputs[key].to(input_device)
        for key in ("input_ids", "attention_mask")
        if key in inputs
    }

    with torch.inference_mode():
        outputs = model.generate(
            **generation_inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    prompt_length = generation_inputs["input_ids"].shape[-1]
    answer_tokens = outputs[0, prompt_length:]

    return tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True
    ).strip()

for i, question in enumerate(TEST_QUESTIONS, start=1):
    print(f"\n{'=' * 65}")
    print(f"TEST {i}: {question}")
    print("-" * 65)
    print(test_helpdeskbuddy(question))

print("\nRetesting completed.")


In [ ]:

# CELL 13: Launch the HelpDeskBuddy browser interface

import torch
import gradio as gr

SYSTEM_PROMPT = (
    "You are HelpDeskBuddy, an IT support troubleshooting assistant. "
    "Give accurate, practical guidance in at most 4 numbered steps. "
    "Use short sentences and finish the answer. "
    "For Outlook, check internet connectivity, Work Offline status, "
    "mailbox storage, filters, and organizational service status. "
    "For DNS, check whether other internal resources work, use "
    "nslookup for the affected internal hostname, and refer DNS "
    "server or record changes to authorized IT staff. "
    "Never invent menu options. Never request passwords, MFA codes, "
    "or secrets. Do not claim actions have been performed. "
    "Refer administrator-only changes to authorized IT staff."
)

def deskbuddy_chat(message, history):
    if not message or not message.strip():
        return "Please describe the IT issue you are experiencing."

    messages = [{
        "role": "user",
        "content": SYSTEM_PROMPT + "\n\nUser issue: " + message.strip()
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    )

    input_device = model.get_input_embeddings().weight.device
    generation_inputs = {
        key: inputs[key].to(input_device)
        for key in ("input_ids", "attention_mask")
        if key in inputs
    }

    try:
        with torch.inference_mode():
            outputs = model.generate(
                **generation_inputs,
                max_new_tokens=180,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        prompt_length = generation_inputs["input_ids"].shape[-1]
        answer = tokenizer.decode(
            outputs[0, prompt_length:],
            skip_special_tokens=True
        ).strip()

        if not answer:
            return (
                "I couldn't generate a useful answer. "
                "Please rephrase the issue or contact your IT support team."
            )

        return answer

    except Exception as exc:
        print("HelpDeskBuddy generation error:", repr(exc))
        return (
            "Sorry, I couldn't process that request. "
            "Please try again or contact your IT support team."
        )

demo = gr.ChatInterface(
    fn=deskbuddy_chat,
    title="HelpDeskBuddy AI Assistant",
    description=(
        "IT support troubleshooting for VPN, Outlook, DNS, Wi-Fi, "
        "Windows, and common workplace IT issues. "
        "Do not enter passwords, MFA codes, or confidential information."
    ),
    examples=[
        "I cannot connect to the office VPN.",
        "Outlook is not receiving new emails.",
        "I cannot access an internal website. How can I check DNS?",
        "My Windows laptop is running slowly."
    ]
)

demo.launch(share=True)


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

PROJECT_DIR = "/content/drive/MyDrive/HelpDeskBuddy"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(
    PROJECT_DIR, "models", "deskbuddy_gemma3_lora"
)
ADAPTER_DIR = os.path.join(MODEL_DIR, "final_adapter")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset folder:", DATA_DIR)
print("Adapter folder:", ADAPTER_DIR)

In [ ]:
%pip install -U \
    transformers \
    datasets \
    accelerate \
    peft \
    trl \
    bitsandbytes \
    gradio \
    huggingface_hub \
    safetensors

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os

PROJECT_DIR = "/content/drive/MyDrive/HelpDeskBuddy"

DATA_DIR = os.path.join(PROJECT_DIR, "data")
MODEL_DIR = os.path.join(
    PROJECT_DIR, "models", "deskbuddy_gemma3_lora"
)
ADAPTER_DIR = os.path.join(MODEL_DIR, "final_adapter")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Dataset folder:", DATA_DIR)
print("Adapter folder:", ADAPTER_DIR)

In [ ]:
import os
import sys
import platform
import torch
import transformers
import peft
import datasets
import gradio

report = f"""
Python: {sys.version}
Platform: {platform.platform()}
PyTorch: {torch.__version__}
Transformers: {transformers.__version__}
PEFT: {peft.__version__}
Datasets: {datasets.__version__}
Gradio: {gradio.__version__}
CUDA available: {torch.cuda.is_available()}
CUDA build: {torch.version.cuda}
GPU count: {torch.cuda.device_count()}
"""

if torch.cuda.is_available():
    report += f"\nGPU: {torch.cuda.get_device_name(0)}"

print(report)

with open(
    os.path.join(PROJECT_DIR, "environment_report.txt"),
    "w",
    encoding="utf-8"
) as f:
    f.write(report)

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
del hf_token

print("Hugging Face authentication completed.")

In [ ]:
import os

os.makedirs(ADAPTER_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Adapter saved to:", ADAPTER_DIR)
print("\nSaved files:")

for filename in sorted(os.listdir(ADAPTER_DIR)):
    print(filename)

In [ ]:

%%writefile /content/requirements.txt
torch
transformers
datasets
peft
trl
accelerate
bitsandbytes
huggingface_hub
safetensors
gradio
PyYAML


In [ ]:

from google.colab import files

files.download("/content/requirements.txt")


In [ ]:

!pip freeze > /content/requirements.txt


In [ ]:

from google.colab import files

files.download("/content/requirements.txt")


In [ ]:
demo.launch(share=True)

In [ ]:

from google.colab import drive
import os

drive.mount('/content/drive')

PROJECT_FOLDER = "/content/drive/MyDrive/HelpDeskBuddy"

os.makedirs(PROJECT_FOLDER, exist_ok=True)

print("Project folder:", PROJECT_FOLDER)
print("Folder created or already exists.")


In [ ]:

import os

PROJECT_FOLDER = "/content/drive/MyDrive/HelpDeskBuddy"
URL_FILE = os.path.join(PROJECT_FOLDER, "gradio_live_url.txt")

# Replace this example with your actual Gradio URL
GRADIO_URL = "https://7f948fbfe0d13095e1.gradio.live/"

with open(URL_FILE, "w", encoding="utf-8") as file:
    file.write(GRADIO_URL + "\n")

print("Gradio URL saved successfully!")
print("File location:", URL_FILE)
print("Saved URL:", GRADIO_URL)


In [ ]:

import os

PROJECT_FOLDER = "/content/drive/MyDrive/HelpDeskBuddy"
os.makedirs(PROJECT_FOLDER, exist_ok=True)

# Launch Gradio and capture its URLs
launch_result = demo.launch(share=True)

local_url = launch_result[1]
share_url = launch_result[2]

print("Local URL:", local_url)
print("Public Gradio URL:", share_url)

# Save the public URL to Google Drive
if share_url:
    url_file = os.path.join(
        PROJECT_FOLDER, "gradio_live_url.txt"
    )

    with open(url_file, "w", encoding="utf-8") as f:
        f.write(share_url + "\n")

    print("URL saved to Google Drive:", url_file)
else:
    print("No public URL returned. Check the Gradio launch output.")


In [ ]:

url_file = "/content/drive/MyDrive/HelpDeskBuddy/gradio_live_url.txt"

with open(url_file, "r", encoding="utf-8") as f:
    saved_url = f.read().strip()

print("Saved HelpDeskBuddy URL:", saved_url)


In [ ]:

from google.colab import drive

drive.mount('/content/drive')


In [ ]:

import os

PROJECT_DIR = "/content/drive/MyDrive/HelpdeskBuddy_AI_Assistant"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder:", PROJECT_DIR)
print("Folder exists:", os.path.isdir(PROJECT_DIR))


In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:

import os

PROJECT_DIR = "/content/drive/MyDrive/HelpDeskBuddy_AI_assistant"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Folder ready:", PROJECT_DIR)


In [ ]:

from google.colab import files

uploaded = files.upload()


In [ ]:

import nbformat
import os

PROJECT_DIR = "/content/HelpdeskBuddy_AI_assistant"

NOTEBOOK_PATH = os.path.join(
    PROJECT_DIR,
    "HelpDeskBuddy_AI_Assistant_For_IT_Support_Troubleshooting.ipynb"
)

os.makedirs(PROJECT_DIR, exist_ok=True)

notebook = nbformat.v4.new_notebook(
    cells=[
        nbformat.v4.new_markdown_cell(
            "# HelpDeskBuddy AI Assistant\n"
            "## IT Support Troubleshooting Chatbot\n\n"
            "This notebook contains the code for the "
            "HelpDeskBuddy AI Assistant project."
        ),
        nbformat.v4.new_code_cell(
            'print("HelpDeskBuddy AI Assistant notebook is ready.")'
        )
    ],
    metadata={
        "kernelspec": {
            "display_name": "Python 3",
            "language": "python",
            "name": "python3"
        }
    }
)

nbformat.write(notebook, NOTEBOOK_PATH)

print("Notebook created successfully!")
print("Location:", NOTEBOOK_PATH)
print("File exists:", os.path.isfile(NOTEBOOK_PATH))


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

DRIVE_ROOT = (
    "/content/drive/MyDrive/"
    "HelpDeskBuddy_AI_assistant"
)

ADAPTER_SOURCE = (
    "/content/deskbuddy_gemma3_lora/final_adapter"
)

ADAPTER_BACKUP = (
    f"{DRIVE_ROOT}/deskbuddy_gemma3_lora/final_adapter"
)

os.makedirs(ADAPTER_BACKUP, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/data/sample", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)

print("Persistent project directories are ready.")
print(DRIVE_ROOT)

In [ ]:
import os
import shutil

assert os.path.isfile(
    os.path.join(ADAPTER_SOURCE, "adapter_config.json")
), "Adapter configuration file not found."

assert os.path.isfile(
    os.path.join(ADAPTER_SOURCE, "adapter_model.safetensors")
), "Adapter weights not found."

shutil.copytree(
    ADAPTER_SOURCE,
    ADAPTER_BACKUP,
    dirs_exist_ok=True
)

print("LoRA adapter copied to Google Drive.")
print("Backup location:", ADAPTER_BACKUP)

In [ ]:
import os

DRIVE_ROOT = (
    "/content/drive/MyDrive/"
    "HelpDeskBuddy_AI_assistant"
)

ADAPTER_SOURCE = (
    "/content/deskbuddy_gemma3_lora/final_adapter"
)

ADAPTER_BACKUP = (
    f"{DRIVE_ROOT}/deskbuddy_gemma3_lora/final_adapter"
)

os.makedirs(ADAPTER_BACKUP, exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/data/sample", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)

print("Persistent project directories are ready.")
print(DRIVE_ROOT)

In [ ]:
import os
import shutil

DATASET_SOURCE = (
    "/content/helpdesk_dataset.jsonl"
)

DATASET_BACKUP = (
    f"{DRIVE_ROOT}/data/sample/helpdesk_dataset.jsonl"
)

if os.path.isfile(DATASET_SOURCE):
    shutil.copy2(DATASET_SOURCE, DATASET_BACKUP)
    print("Dataset backup completed:", DATASET_BACKUP)
else:
    print("Dataset not found at the assumed path.")
    print("Check the actual dataset location before copying.")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=(
        "/content/drive/MyDrive/"
        "HelpDeskBuddy_AI_assistant/checkpoints"
    ),
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    logging_steps=10,
    report_to="none",
)

In [ ]:
import os

for root, dirs, files in os.walk(DRIVE_ROOT):
    for filename in files:
        path = os.path.join(root, filename)
        print(path)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:

import os

drive_root = "/content/drive/MyDrive"

target_names = {
    "training.yaml",
    "training_config.yaml",
    "requirements.txt",
}

found = []

for root, dirs, files in os.walk(drive_root):
    # Skip hidden folders
    dirs[:] = [d for d in dirs if not d.startswith(".")]

    for filename in files:
        if filename.lower() in target_names:
            found.append(os.path.join(root, filename))

if found:
    print("Files found in Google Drive:")
    for path in found:
        print(path)
else:
    print("No matching files found in My Drive.")
    print("The files may not exist yet or may be elsewhere.")


In [ ]:

import os

for root, dirs, files in os.walk("/content"):
    dirs[:] = [
        d for d in dirs
        if d not in {"drive", ".cache", "__pycache__"}
    ]

    for filename in files:
        if filename.lower() in {
            "training.yaml",
            "training_config.yaml",
            "requirements.txt",
        }:
            print(os.path.join(root, filename))


In [ ]:
import os
import shutil

assert os.path.isfile(
    os.path.join(ADAPTER_SOURCE, "adapter_config.json")
), "Adapter configuration file not found."

assert os.path.isfile(
    os.path.join(ADAPTER_SOURCE, "adapter_model.safetensors")
), "Adapter weights not found."

shutil.copytree(
    ADAPTER_SOURCE,
    ADAPTER_BACKUP,
    dirs_exist_ok=True
)

print("LoRA adapter copied to Google Drive.")
print("Backup location:", ADAPTER_BACKUP)

In [ ]:
import os

for root, dirs, files in os.walk(DRIVE_ROOT):
    for filename in files:
        path = os.path.join(root, filename)
        print(path)

In [ ]:

from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/HelpDeskBuddy_AI_assistant"
)

CONFIG_DIR = PROJECT_DIR / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_PATH = CONFIG_DIR / "training_config.yaml"

print("Configuration file location:", CONFIG_PATH)


In [ ]:

yaml_content = """# HelpDeskBuddy AI Assistant Training Configuration

model:
  model_id: "google/gemma-3-4b-it"
  max_length: 512

dataset:
  train_file: "data/sample/helpdesk_dataset.jsonl"

lora:
  r: 8
  alpha: 16
  dropout: 0.05

training:
  per_device_train_batch_size: 1
  gradient_accumulation_steps: 4
  learning_rate: 0.0001
  num_train_epochs: 1
  logging_steps: 10
  save_steps: 25

output:
  adapter_dir: "outputs/deskbuddy_lora"
"""

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    f.write(yaml_content)

print("Created and saved:", CONFIG_PATH)


In [ ]:

assert CONFIG_PATH.is_file(), "Configuration file was not created."

print(CONFIG_PATH.read_text(encoding="utf-8"))
print("\nFile size:", CONFIG_PATH.stat().st_size, "bytes")


In [ ]:
%pip install pyyaml

In [ ]:

import yaml

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Base model:", config["model"]["model_id"])
print("Maximum sequence length:", config["model"]["max_length"])
print("LoRA rank:", config["lora"]["r"])
print("LoRA alpha:", config["lora"]["alpha"])
print("Learning rate:", config["training"]["learning_rate"])
print("Epochs:", config["training"]["num_train_epochs"])


In [ ]:

import os
import shutil
from pathlib import Path
from datetime import datetime

DRIVE_ROOT = Path("/content/drive/MyDrive")
DEST_DIR = DRIVE_ROOT / "HelpDeskBuddy_AI_assistant"
DEST_DIR.mkdir(parents=True, exist_ok=True)

DEST_FILE = (
    DEST_DIR /
    "HelpDeskBuddy_AI_Assistant_For_IT_Support_Troubleshooting.ipynb"
)

matches = []

for root, dirs, files in os.walk(DRIVE_ROOT):
    # Avoid searching trash and hidden folders
    dirs[:] = [
        d for d in dirs
        if not d.startswith(".") and d != ".Trash"
    ]

    for filename in files:
        lower = filename.lower()

        if (
            lower.endswith(".ipynb")
            and ("helpdeskbuddy" in lower or "helpdesk" in lower)
            and Path(root) / filename != DEST_FILE
        ):
            source = Path(root) / filename
            matches.append(source)

if not matches:
    print("No matching notebook files found in Google Drive.")
    print("Save your latest Colab notebook to Drive first.")
else:
    # Select the file with the latest filesystem modification time
    matches.sort(
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )

    print("Notebook files found:")
    for i, path in enumerate(matches, start=1):
        modified = datetime.fromtimestamp(
            path.stat().st_mtime
        ).strftime("%Y-%m-%d %H:%M:%S")

        print(f"{i}. {path}")
        print(f"   Modified: {modified}")

    latest = matches[0]

    print("\nSelected source:", latest)

    # Copy the newest file to your permanent project folder
    shutil.copy2(latest, DEST_FILE)

    print("\nCopy completed successfully.")
    print("Destination:", DEST_FILE)
    print("File size:", DEST_FILE.stat().st_size, "bytes")


In [ ]:

import json

NOTEBOOK_PATH = (
    "/content/drive/MyDrive/"
    "HelpDeskBuddy_AI_assistant/"
    "HelpDeskBuddy_AI_Assistant_For_IT_Support_Troubleshooting.ipynb"
)

with open(NOTEBOOK_PATH, "r", encoding="utf-8") as f:
    notebook = json.load(f)

print("Notebook file exists:", os.path.isfile(NOTEBOOK_PATH))
print("Notebook format version:", notebook.get("nbformat"))
print("Number of cells:", len(notebook.get("cells", [])))
print("Saved location:", NOTEBOOK_PATH)
